# 9 · Nonlinear problems — Allen–Cahn & Newton

Until now every system was **linear** — one solve and done. Many real models are **nonlinear**:
the operator depends on the unknown itself. The workhorse is **Newton's method** — linearise,
solve, repeat — and NGSolve builds the needed **Jacobian automatically** by differentiating the
weak form. We meet it on the **Allen–Cahn** equation, a phase-separation flow.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from ngsolve import solvers
from netgen.geom2d import SplineGeometry
from ngsolve.webgui import Draw
import numpy as np, random
import matplotlib.pyplot as plt

## 1. The Allen–Cahn model

A field $u(t,\mathbf x)\in[-1,1]$ labels two **phases** ($u=\pm1$). It flows downhill in the
**Ginzburg–Landau energy** $E(u)=\int \tfrac\gamma2|\nabla u|^2+\tfrac1\gamma W(u)$, with the
**double-well** $W(u)=\tfrac14(u^2-1)^2$ (two minima at $\pm1$). The gradient flow is
$$ \partial_t u \;=\; \gamma\,\Delta u \;+\; \tfrac1\gamma\,(u-u^3) , $$
(no-flux boundaries). The reaction $u-u^3=-W'(u)$ pushes $u$ toward $\pm1$; the $\gamma\Delta u$
smooths interfaces of width $\sim\gamma$. The **$u^3$ is the nonlinearity** — that is what makes
the implicit step a *nonlinear* solve.

In [ ]:
geo = SplineGeometry(); geo.AddRectangle((0, 0), (1, 1))
mesh = Mesh(geo.GenerateMesh(maxh=0.025))
gamma, dt = 0.03, 0.002                                # interface width ~gamma, time step
fes = H1(mesh, order=2)                                 # no-flux (Neumann) -> no Dirichlet dofs
u, v = fes.TnT()
gfu = GridFunction(fes); uold = GridFunction(fes)
random.seed(1)                                         # a tiny random seed -> spinodal decomposition
for i in range(len(gfu.vec)):
    gfu.vec[i] = 0.1*(random.random() - 0.5)
Draw(gfu, mesh, "phase u", min=-1, max=1, autoscale=False)

## 2. One implicit step is nonlinear — solve it with Newton

Implicit Euler for the step $u^n\to u^{n+1}\equiv u$ gives the **residual**
$$ F(u)\;=\;\int \frac{u-u^n}{\Delta t}\,v \;+\; \gamma\,\nabla u\!\cdot\!\nabla v
   \;-\; \tfrac1\gamma(u-u^3)\,v \;\;\mathrm dx \;\overset!=\;0 \quad\forall v, $$
**nonlinear in $u$** through the $u^3$. We write $F$ as a `BilinearForm` (the trial function `u`
*is* the unknown), and **`solvers.Newton`** does the rest: at each iterate it assembles the
**Jacobian** $F'(u)$ — NGSolve differentiates the form **automatically** — solves $F'(u)\,\delta
= -F(u)$, updates $u\mathrel{+}=\delta$, and repeats until $\|F\|$ is tiny (a few iterations).

In [ ]:
a = BilinearForm(fes)
a += (1/dt)*(u - uold)*v*dx + gamma*grad(u)*grad(v)*dx - (1/gamma)*(u - u**3)*v*dx

uold.vec.data = gfu.vec
solvers.Newton(a, gfu, printing=True, maxit=20)        # ONE nonlinear step, Newton iterations shown
print(f"after one step: u in [{min(gfu.vec):.3f}, {max(gfu.vec):.3f}]")

## 3. March in time — watch the phases coarsen

Repeat the nonlinear step. From the random seed the field **separates** into $\pm1$ patches
(spinodal decomposition), then the patches **coarsen** — interfaces shrink to lower the energy.

In [ ]:
nsteps = 150
gx = gy = np.linspace(0, 1, 90); mips = [mesh(float(xx), float(yy)) for yy in gy for xx in gx]
frames, energy = [], []
with TaskManager():
    for step in range(1, nsteps+1):
        uold.vec.data = gfu.vec
        solvers.Newton(a, gfu, printing=False, maxit=20)
        if step % 10 == 0:
            frames.append(np.array([gfu(mp) for mp in mips]).reshape(len(gy), len(gx)))
            energy.append(Integrate(0.5*gamma*grad(gfu)*grad(gfu) + (1/gamma)*0.25*(gfu*gfu-1)**2, mesh))
print(f"stepped to t={nsteps*dt:.2f}:  u in [{min(gfu.vec):.2f}, {max(gfu.vec):.2f}],  {len(frames)} frames")

In [ ]:
from matplotlib import animation
from IPython.display import HTML
fig, ax = plt.subplots(figsize=(4.4, 4.4))
def draw(i):
    ax.clear()
    ax.contourf(gx, gy, frames[i], levels=np.linspace(-1, 1, 21), cmap="coolwarm")
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"t = {10*(i+1)*dt:.2f}")
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=120)
plt.close(fig)
HTML(anim.to_jshtml())

The energy **decreases monotonically** — the whole point of a gradient flow:

In [ ]:
plt.figure(figsize=(7, 2.6))
plt.plot([10*(i+1)*dt for i in range(len(energy))], energy, "-o", ms=3)
plt.xlabel("time"); plt.ylabel("Ginzburg–Landau energy"); plt.grid(alpha=0.3)
plt.title("energy decreases — a gradient flow"); plt.tight_layout()

**Next:** combine **unsteady** (unit 8) and **nonlinear** (this unit) and add a *second* reacting
species — and patterns grow themselves on the Beast's skin (Part III, Turing).

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("08-unsteady-doubleglazing", "8 · Unsteady problems — the double-glazing flow")
    _next = ("10-dg-hdg", "10 · Discontinuous Galerkin & HDG")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))